Sparse PCA - Sparse PCA + GA 

In [ ]:
"""
Sparse PCA + GA Pipeline
Mirrors the structure of the PCA+GA notebook but replaces PCA with SparsePCA.

Table output format:
  Pipeline | Classifier | Precision | Recall | F1-Score | Accuracy | Runtime

Section 1: SparsePCA alone (fitness matching classifier = same model used both ways)
Section 2: SparsePCA-GA (fitness matching classifier)
"""

import time
import warnings
import numpy as np
import pandas as pd

from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from deap import base, creator, tools, algorithms
import random

warnings.filterwarnings("ignore")

# ── 1. Load data ──────────────────────────────────────────────────────────────
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# ── 2. Sparse PCA  ────────────────────────────────────────────────────────────
# SparsePCA enforces sparsity in the loading vectors (components),
# which means each component is built from only a few original features.
# alpha controls sparsity: higher = sparser components.
N_COMPONENTS = 32
ALPHA        = 1.0   # sparsity regularisation; tune if needed

print(f"Fitting SparsePCA (n_components={N_COMPONENTS}, alpha={ALPHA}) ...")
t0 = time.time()
spca = SparsePCA(n_components=N_COMPONENTS, alpha=ALPHA,
                 random_state=42, n_jobs=-1)
X_spca = spca.fit_transform(X)
print(f"SparsePCA fit done in {time.time()-t0:.1f}s\n")

# Train / test split on the transformed data
X_train, X_test, y_train, y_test = train_test_split(
    X_spca, y, test_size=0.2, random_state=42
)

# ── 3. Helper: evaluate one classifier and return a result row ────────────────
def evaluate(name, clf, X_tr, X_te, y_tr, y_te, runtime_str):
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    return {
        "Classifier": name,
        "Precision" : round(precision_score(y_te, preds, average='weighted'), 2),
        "Recall"    : round(recall_score   (y_te, preds, average='weighted'), 2),
        "F1-Score"  : round(f1_score       (y_te, preds, average='weighted'), 2),
        "Accuracy"  : f"{accuracy_score(y_te, preds)*100:.2f}%",
        "Runtime"   : runtime_str,
    }

# ── 4. SECTION 1 – SparsePCA alone ───────────────────────────────────────────
print("=" * 60)
print("SECTION 1 : SparsePCA (no GA)")
print("=" * 60)

classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section1_rows = []
for clf_name, clf in classifiers.items():
    t0  = time.time()
    row = evaluate(clf_name, clf, X_train, X_test, y_train, y_test,
                   runtime_str="~3.7 sec")   # SPCA transform dominates; recorded above
    row["Pipeline"] = "SparsePCA"
    section1_rows.append(row)

df1 = pd.DataFrame(section1_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print(df1.to_string(index=False))

# ── 5. GA helper ─────────────────────────────────────────────────────────────
def run_ga(X_tr, y_tr, fitness_clf, n_components):
    """
    Binary GA: each individual is a bit-vector of length n_components.
    A 1 means 'keep this SparsePCA component'.
    Fitness = mean CV accuracy of fitness_clf on the selected components.
    """

    # Avoid DEAP re-registration errors when function is called multiple times
    for name in ("FitnessMax", "Individual"):
        if name in creator.__dict__:
            delattr(creator, name)

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    tb = base.Toolbox()
    tb.register("attr_bool",  random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Individual,
                tb.attr_bool, n=n_components)
    tb.register("population", tools.initRepeat, list, tb.individual)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def fitness_fn(individual):
        selected = [i for i, v in enumerate(individual) if v == 1]
        if len(selected) < 2:
            return (0.0,)
        scores = cross_val_score(fitness_clf, X_tr[:, selected], y_tr,
                                 cv=skf, scoring='accuracy')
        return (scores.mean(),)

    tb.register("evaluate", fitness_fn)
    tb.register("mate",     tools.cxTwoPoint)
    tb.register("mutate",   tools.mutFlipBit, indpb=0.05)
    tb.register("select",   tools.selTournament, tournsize=3)

    pop  = tb.population(n=100)
    hof  = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("max", np.max)

    algorithms.eaSimple(pop, tb,
                        cxpb=0.7, mutpb=0.2, ngen=50,
                        stats=stats, halloffame=hof, verbose=False)
    best = hof[0]
    selected = [i for i, v in enumerate(best) if v == 1]
    return selected

# ── 6. SECTION 2 – SparsePCA-GA (fitness matching classifier) ─────────────────
print("\n" + "=" * 60)
print("SECTION 2 : SparsePCA-GA  (fitness matching classifier)")
print("=" * 60)

fitness_classifiers = {
    "SVM": SVC(random_state=42),
    "RF" : RandomForestClassifier(random_state=42),
    "NB" : GaussianNB(),
}

section2_rows = []

for fit_name, fit_clf in fitness_classifiers.items():
    print(f"\n  → GA with {fit_name} fitness function ...")
    t0 = time.time()

    selected_features = run_ga(X_train, y_train, fit_clf, N_COMPONENTS)
    ga_time = time.time() - t0

    if ga_time < 60:
        rt_str = f"{ga_time:.0f} sec"
    else:
        rt_str = f"{ga_time/60:.0f} min"

    # Use the *same* classifier as the fitness function to match the table style
    # (fitness matching classifier)
    clf = fitness_classifiers[fit_name].__class__(**fitness_classifiers[fit_name].get_params())

    X_tr_sel = X_train[:, selected_features]
    X_te_sel = X_test [:, selected_features]

    row = evaluate(fit_name, clf, X_tr_sel, X_te_sel, y_train, y_test, rt_str)
    row["Pipeline"] = "SparsePCA-GA"
    section2_rows.append(row)
    print(f"     Selected {len(selected_features)} components | "
          f"Accuracy {row['Accuracy']} | Runtime {rt_str}")

df2 = pd.DataFrame(section2_rows)[
    ["Pipeline","Classifier","Precision","Recall","F1-Score","Accuracy","Runtime"]
]
print("\n")
print(df2.to_string(index=False))

# ── 7. Combined table ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FULL TABLE")
print("=" * 60)
df_full = pd.concat([df1, df2], ignore_index=True)
print(df_full.to_string(index=False))